In [1]:
# Load engineered data (rebuild from raw CSV via src.feature_engineering if missing)
from pathlib import Path
import sys

_root = Path("..").resolve()
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from src.feature_engineering import classify_signal, load_engineered_frames

btc_df, ada_df = load_engineered_frames()
print(f" Loaded engineered data: BTC {btc_df.shape}, ADA {ada_df.shape}")

 Loaded engineered data: BTC (2344, 49), ADA (2344, 42)


In [2]:
from src.feature_engineering import get_feature_columns

# NOTE: We do NOT scale here. Scaling on the full dataframe before walk-forward
# would leak the mean/std of future windows into earlier folds. Each fold in
# walk_forward_validation now fits its own StandardScaler on train only.
btc_feature_cols = get_feature_columns(btc_df)
ada_feature_cols = get_feature_columns(ada_df)
print(f"Feature columns: BTC={len(btc_feature_cols)}, ADA={len(ada_feature_cols)}")

Feature columns: BTC=43, ADA=36


# Logistic Regression Pipeline

Walk-forward validation with leak-free per-fold `StandardScaler` and `TimeSeriesSplit` inside `RandomizedSearchCV`. Signals: Buy (+1) / Hold (0) / Sell (-1) via `classify_signal(threshold)`. Features from `src.feature_engineering.get_feature_columns`.

In [3]:
# =============================================================================
# SECTION 1: IMPORTS & DEPENDENCIES
# =============================================================================
import requests
from datetime import datetime, timedelta
import time
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
import numpy as np
import pandas as pd

# Shared config
horizons = [3, 7, 14, 30]
thresholds = {
    "fixed_0.5%": 0.005,
    "fixed_1%": 0.01,
    "fixed_2%": 0.02,
}

# ML Libraries
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support, accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import RandomizedSearchCV

In [4]:
from sklearn.model_selection import TimeSeriesSplit
from src.feature_engineering import BINARY_COLUMNS


def _split_scale_columns(X):
    """Return (continuous_idx, binary_idx) column indices; binary columns are not scaled."""
    if hasattr(X, "columns"):
        cols = list(X.columns)
        binary_idx = np.array([i for i, c in enumerate(cols) if c in BINARY_COLUMNS], dtype=int)
        cont_idx = np.array([i for i, c in enumerate(cols) if c not in BINARY_COLUMNS], dtype=int)
    else:
        cont_idx = np.arange(X.shape[1], dtype=int)
        binary_idx = np.array([], dtype=int)
    return cont_idx, binary_idx


def _to_np(X):
    return X.to_numpy() if hasattr(X, "to_numpy") else np.asarray(X)


def _scale_train_test(X_train_np, X_test_np, cont_idx):
    """Fit StandardScaler on training continuous columns only, then transform both."""
    if len(cont_idx) == 0:
        return X_train_np, X_test_np
    scaler = StandardScaler()
    X_train_scaled = X_train_np.astype(np.float64, copy=True)
    X_test_scaled = X_test_np.astype(np.float64, copy=True)
    X_train_scaled[:, cont_idx] = scaler.fit_transform(X_train_np[:, cont_idx])
    X_test_scaled[:, cont_idx] = scaler.transform(X_test_np[:, cont_idx])
    return X_train_scaled, X_test_scaled


def walk_forward_validation(X, y, model_instance, param_dist, initial_train_size=0.6, step=30, n_iter=2, cv=3, scoring='f1_macro'):
    """Walk-forward validation with per-fold scaling and TimeSeriesSplit inner CV.

    Phase 0: also returns per-fold std of F1 (std_f1_*) so a single mean is never
    reported on its own. A std comparable to the mean signals an unstable signal.
    """
    cont_idx, _ = _split_scale_columns(X)
    X_np = _to_np(X)
    y_np = np.asarray(y)
    n = len(X_np)
    train_end = int(initial_train_size * n)
    results = []
    for start in range(train_end, n, step):
        X_train_np, X_test_np = X_np[:start], X_np[start:start + step]
        y_train_np, y_test_np = y_np[:start], y_np[start:start + step]
        if len(X_test_np) == 0:
            break
        X_train_np, X_test_np = _scale_train_test(X_train_np, X_test_np, cont_idx)
        inner_cv = TimeSeriesSplit(n_splits=cv)
        search = RandomizedSearchCV(model_instance, param_dist, n_iter=n_iter, cv=inner_cv, scoring=scoring, random_state=42)
        search.fit(X_train_np, y_train_np)
        model = search.best_estimator_
        y_pred = model.predict(X_test_np)
        precision, recall, fscore, _ = precision_recall_fscore_support(y_test_np, y_pred, labels=[-1, 0, 1], zero_division=0)
        results.append({
            'precision_sell': precision[0], 'recall_sell': recall[0], 'f1_sell': fscore[0],
            'precision_hold': precision[1], 'recall_hold': recall[1], 'f1_hold': fscore[1],
            'precision_buy':  precision[2], 'recall_buy':  recall[2], 'f1_buy':  fscore[2],
        })
    if not results:
        return {k: 0 for k in ['avg_precision_sell','avg_recall_sell','avg_f1_sell','avg_precision_hold','avg_recall_hold','avg_f1_hold','avg_precision_buy','avg_recall_buy','avg_f1_buy','std_f1_sell','std_f1_hold','std_f1_buy','n_folds']}
    return {
        'avg_precision_sell': float(np.mean([r['precision_sell'] for r in results])),
        'avg_recall_sell':    float(np.mean([r['recall_sell']    for r in results])),
        'avg_f1_sell':        float(np.mean([r['f1_sell']        for r in results])),
        'avg_precision_hold': float(np.mean([r['precision_hold'] for r in results])),
        'avg_recall_hold':    float(np.mean([r['recall_hold']    for r in results])),
        'avg_f1_hold':        float(np.mean([r['f1_hold']        for r in results])),
        'avg_precision_buy':  float(np.mean([r['precision_buy']  for r in results])),
        'avg_recall_buy':     float(np.mean([r['recall_buy']     for r in results])),
        'avg_f1_buy':         float(np.mean([r['f1_buy']         for r in results])),
        'std_f1_sell':        float(np.std([r['f1_sell']         for r in results])),
        'std_f1_hold':        float(np.std([r['f1_hold']         for r in results])),
        'std_f1_buy':         float(np.std([r['f1_buy']          for r in results])),
        'n_folds':            int(len(results)),
    }


def walk_forward_with_predictions(X, y, model_instance, param_dist, initial_train_size=0.6, step=30, n_iter=2, cv=3, scoring='f1_macro'):
    """Walk-forward variant that returns aggregated predictions for Binary MCC."""
    cont_idx, _ = _split_scale_columns(X)
    X_np = _to_np(X)
    y_np = np.asarray(y)
    fold_metrics = defaultdict(list)
    all_y_true, all_y_pred = [], []
    n = len(X_np)
    idx = int(initial_train_size * n)
    fold_count = 0
    while idx < n:
        X_train_np, X_test_np = X_np[:idx], X_np[idx:idx + step]
        y_train_np, y_test_np = y_np[:idx], y_np[idx:idx + step]
        if len(X_test_np) == 0:
            break
        X_train_np, X_test_np = _scale_train_test(X_train_np, X_test_np, cont_idx)
        inner_cv = TimeSeriesSplit(n_splits=cv)
        search = RandomizedSearchCV(model_instance, param_dist, n_iter=n_iter, cv=inner_cv, scoring=scoring, random_state=42)
        try:
            search.fit(X_train_np, y_train_np)
            model = search.best_estimator_
        except Exception as e:
            print(f"Fold {fold_count + 1} failed: {e}")
            idx += step
            continue
        y_pred_fold = model.predict(X_test_np)
        all_y_true.extend(y_test_np.tolist())
        all_y_pred.extend(y_pred_fold.tolist())
        prec, rec, f1, _ = precision_recall_fscore_support(y_test_np, y_pred_fold, labels=[-1, 0, 1], average=None, zero_division=0)
        for i, label_name in enumerate(["sell", "hold", "buy"]):
            fold_metrics[f'precision_{label_name}'].append(prec[i])
            fold_metrics[f'recall_{label_name}'].append(rec[i])
            fold_metrics[f'f1_{label_name}'].append(f1[i])
        fold_count += 1
        idx += step
    avg_metrics = {key: np.mean(values) for key, values in fold_metrics.items()}
    return avg_metrics, all_y_true, all_y_pred

In [5]:
# ============================================================================
# Simple MCC utilities (standard + weighted)
# ============================================================================
def _mcc_from_counts(tp, tn, fp, fn):
    import math
    denom = (tp+fp)*(tp+fn)*(tn+fp)*(tn+fn)
    if denom == 0:
        return 0.0
    return ((tp*tn) - (fp*fn)) / math.sqrt(denom)


def evaluate_signal_quality(y_true, y_pred, verbose=True, opposite_weight=2.0):
    import numpy as np
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    def eval_side(pos_label):
        # Binary mapping for MCC
        y_tb = (y_true == pos_label).astype(int)
        y_pb = (y_pred == pos_label).astype(int)
        tp = int(((y_tb == 1) & (y_pb == 1)).sum())
        tn = int(((y_tb == 0) & (y_pb == 0)).sum())
        fp = int(((y_tb == 0) & (y_pb == 1)).sum())
        fn = int(((y_tb == 1) & (y_pb == 0)).sum())
        mcc = _mcc_from_counts(tp, tn, fp, fn)

        # Weighted errors: opposite-direction gets higher penalty
        if pos_label == 1:
            opposite_fp = int(((y_true == -1) & (y_pred == 1)).sum())  # predicted Buy when true Sell
            opposite_fn = int(((y_true == 1) & (y_pred == -1)).sum())  # predicted Sell when true Buy
            miss_from_hold = int(((y_true == 1) & (y_pred == 0)).sum())
            false_from_hold = int(((y_true == 0) & (y_pred == 1)).sum())
        else:  # pos_label == -1 (Sell)
            opposite_fp = int(((y_true == 1) & (y_pred == -1)).sum())  # predicted Sell when true Buy
            opposite_fn = int(((y_true == -1) & (y_pred == 1)).sum())  # predicted Buy when true Sell
            miss_from_hold = int(((y_true == -1) & (y_pred == 0)).sum())
            false_from_hold = int(((y_true == 0) & (y_pred == -1)).sum())

        w_fp = opposite_weight*opposite_fp + false_from_hold
        w_fn = opposite_weight*opposite_fn + miss_from_hold
        w_mcc = _mcc_from_counts(tp, tn, w_fp, w_fn)

        details = {
            'tp': tp, 'tn': tn, 'fp': fp, 'fn': fn,
            'opposite_fp': opposite_fp, 'opposite_fn': opposite_fn,
            'miss_from_hold': miss_from_hold, 'false_from_hold': false_from_hold,
            'mcc': round(mcc, 4), 'weighted_mcc': round(w_mcc, 4)
        }
        return details

    buy = eval_side(1)
    sell = eval_side(-1)

    if verbose:
        total = len(y_true)
        print("\n============================================================")
        print("Binary MCC for Buy Signal (Class 1)")
        print("============================================================")
        print(f"MCC: {buy['mcc']}")
        print("Weighted MCC components (Buy):")
        print(f"  [OK] TP: {buy['tp']}, [OK] TN: {buy['tn']}")
        print(f"  [X] Opposite FP (pred Buy | true Sell): {buy['opposite_fp']} (x{opposite_weight})")
        print(f"  [X] Opposite FN (pred Sell | true Buy): {buy['opposite_fn']} (x{opposite_weight})")
        print(f"  [X] Miss from Hold (true Buy | pred Hold): {buy['miss_from_hold']}")
        print(f"  [X] False Buy from Hold (true Hold | pred Buy): {buy['false_from_hold']}")
        print(f"Weighted MCC (Buy): {buy['weighted_mcc']}")

        print("\n============================================================")
        print("Binary MCC for Sell Signal (Class -1)")
        print("============================================================")
        print(f"MCC: {sell['mcc']}")
        print("Weighted MCC components (Sell):")
        print(f"  [OK] TP: {sell['tp']}, [OK] TN: {sell['tn']}")
        print(f"  [X] Opposite FP (pred Sell | true Buy): {sell['opposite_fp']} (x{opposite_weight})")
        print(f"  [X] Opposite FN (pred Buy | true Sell): {sell['opposite_fn']} (x{opposite_weight})")
        print(f"  [X] Miss from Hold (true Sell | pred Hold): {sell['miss_from_hold']}")
        print(f"  [X] False Sell from Hold (true Hold | pred Sell): {sell['false_from_hold']}")
        print(f"Weighted MCC (Sell): {sell['weighted_mcc']}")

    return {
        'buy_mcc': buy['mcc'],
        'buy_weighted_mcc': buy['weighted_mcc'],
        'sell_mcc': sell['mcc'],
        'sell_weighted_mcc': sell['weighted_mcc'],
    }

## Binary MCC Evaluation

**Buy/Sell signals** evaluated as independent binary classifiers. MCC (-1 to +1) handles class imbalance better than accuracy or F1 alone. The **weighted variant** penalises opposite-direction errors (Buy predicted when true is Sell, or vice versa) at 2x relative to missed signals into Hold.

In [6]:
%%time
# ============================================================================
# SECTION 5A: BITCOIN (BTC) - WALK-FORWARD VALIDATION
# ============================================================================
btc_results = []

lr_model = LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced')
lr_param_dist = {
    'C': [0.1, 1, 10],
    'solver': ['lbfgs']
}

for h in horizons:
    for name, t in thresholds.items():
        threshold = t
        btc_df[f"signal_{h}d_{name}"] = btc_df[f"fwd_return_{h}d"].apply(lambda x: classify_signal(x, threshold))
        feature_cols = get_feature_columns(btc_df)
        X = btc_df[feature_cols]
        y = btc_df[f"signal_{h}d_{name}"]
        combined = pd.concat([X, y], axis=1).dropna()
        X_clean = combined[feature_cols]
        y_clean = combined[f"signal_{h}d_{name}"].astype(int)
        if len(X_clean) < 100:
            print(f"BTC {h}d {name}: Only {len(X_clean)} samples, skipping...")
            continue
        avg_metrics = walk_forward_validation(X_clean, y_clean, lr_model, lr_param_dist, cv=3)
        btc_results.append({
            'crypto': 'BTC',
            'horizon_days': h,
            'threshold_type': name,
            'threshold_value': round(threshold, 4),
            **avg_metrics
        })

btc_results_df = pd.DataFrame(btc_results)

CPU times: total: 1min 54s
Wall time: 1min 49s


In [7]:
%%time
# ============================================================================
# SECTION 5B: CARDANO (ADA) - WALK-FORWARD VALIDATION
# ============================================================================
ada_results = []

lr_model = LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced')
lr_param_dist = {
    'C': [0.1, 1, 10],
    'solver': ['lbfgs']
}

for h in horizons:
    for name, t in thresholds.items():
        threshold = t
        ada_df[f"signal_{h}d_{name}"] = ada_df[f"fwd_return_{h}d"].apply(lambda x: classify_signal(x, threshold))
        feature_cols = get_feature_columns(ada_df)
        X = ada_df[feature_cols]
        y = ada_df[f"signal_{h}d_{name}"]
        combined = pd.concat([X, y], axis=1).dropna()
        X_clean = combined[feature_cols]
        y_clean = combined[f"signal_{h}d_{name}"].astype(int)
        if len(X_clean) < 100:
            print(f"ADA {h}d {name}: Only {len(X_clean)} samples, skipping...")
            continue
        avg_metrics = walk_forward_validation(X_clean, y_clean, lr_model, lr_param_dist, cv=3)
        ada_results.append({
            'crypto': 'ADA',
            'horizon_days': h,
            'threshold_type': name,
            'threshold_value': round(threshold, 4),
            **avg_metrics
        })

ada_results_df = pd.DataFrame(ada_results)

CPU times: total: 2min 18s
Wall time: 2min 11s


In [8]:
# ============================================================================
# SECTION 6: COMBINED RESULTS & SUMMARY
# ============================================================================
# Merge BTC and ADA results for comprehensive comparison
# Sorted by cryptocurrency, time horizon, and threshold value for readability
# ============================================================================

# Combine results from both cryptocurrencies
all_results_df = pd.concat([btc_results_df, ada_results_df], ignore_index=True)

# Sort for clear presentation
# - crypto: Group Bitcoin and Cardano separately
# - horizon_days: Compare results across time horizons
# - threshold_value: Compare results across threshold configurations
all_results_df.sort_values(['crypto', 'horizon_days', 'threshold_value'], inplace=True)

print("\n" + "="*80)
print("LOGISTIC REGRESSION MODEL RESULTS")
print("="*80)
print(f"Total Configurations Evaluated: {len(all_results_df)}")
print(f"  - Cryptocurrencies: 2 (BTC, ADA)")
print(f"  - Horizons: 4 (3, 7, 14, 30 days)")
print(f"  - Thresholds: 3 (fixed_0.5%, fixed_1%, fixed_2%)")
print("="*80 + "\n")

all_results_df


LOGISTIC REGRESSION MODEL RESULTS
Total Configurations Evaluated: 24
  - Cryptocurrencies: 2 (BTC, ADA)
  - Horizons: 4 (3, 7, 14, 30 days)
  - Thresholds: 3 (fixed_0.5%, fixed_1%, fixed_2%)



,crypto,horizon_days,threshold_type,threshold_value,avg_precision_sell,avg_recall_sell,avg_f1_sell,avg_precision_hold,avg_recall_hold,avg_f1_hold,avg_precision_buy,avg_recall_buy,avg_f1_buy,std_f1_sell,std_f1_hold,std_f1_buy,n_folds
12,ADA,3,fixed_0.5%,0.005,0.491462,0.457026,0.426419,0.052727,0.273563,0.081786,0.422363,0.231604,0.242358,0.208078,0.138345,0.210514,29
13,ADA,3,fixed_1%,0.010,0.467495,0.427078,0.373829,0.095443,0.264286,0.123830,0.355626,0.286154,0.252019,0.215013,0.168947,0.200025,29
14,ADA,3,fixed_2%,0.020,0.459558,0.465785,0.404778,0.196059,0.391974,0.249404,0.206441,0.157934,0.145094,0.175712,0.199068,0.160600,29
15,ADA,7,fixed_0.5%,0.005,0.413802,0.227534,0.232921,0.044003,0.318966,0.072757,0.235949,0.246665,0.203290,0.223088,0.106295,0.216650,29
16,ADA,7,fixed_1%,0.010,0.386938,0.185350,0.211917,0.095288,0.489943,0.149878,0.180355,0.142189,0.133943,0.198665,0.128025,0.204159,29
17,ADA,7,fixed_2%,0.020,0.385473,0.231240,0.234742,0.147454,0.522226,0.211692,0.221127,0.143357,0.150238,0.200551,0.177474,0.210495,29
18,ADA,14,fixed_0.5%,0.005,0.508262,0.358853,0.357475,0.046181,0.364943,0.073350,0.178041,0.096079,0.087545,0.270990,0.114807,0.170037,29
19,ADA,14,fixed_1%,0.010,0.504845,0.410921,0.382796,0.059418,0.314943,0.090648,0.113156,0.095119,0.082248,0.242285,0.126537,0.183017,29
20,ADA,14,fixed_2%,0.020,0.528440,0.452137,0.398716,0.143422,0.356486,0.175036,0.187211,0.121957,0.109598,0.254227,0.194237,0.202655,29
21,ADA,30,fixed_0.5%,0.005,0.477411,0.345479,0.354530,0.025051,0.244048,0.039934,0.261171,0.147166,0.162274,0.328325,0.073091,0.291440,28


In [9]:
# ============================================================================
# PHASE 0: 24-config averages + spread (never a single number)
# ============================================================================
print("=" * 80)
print("PHASE 0 SUMMARY - LR, 24 configurations")
print("=" * 80)
for side in ["buy", "sell"]:
    m = all_results_df[f"avg_f1_{side}"].mean()
    s = all_results_df[f"avg_f1_{side}"].std()
    fold = all_results_df[f"std_f1_{side}"].median()
    print(f"{side.title():4} F1: {m:.4f} +/- {s:.4f} across configs | "
          f"median within-config fold-std {fold:.4f}")

best = all_results_df.loc[all_results_df["avg_f1_sell"].idxmax()]
print(f"\nBest Sell-F1 config: {best['crypto']} {int(best['horizon_days'])}d "
      f"{best['threshold_type']} -> Sell F1 {best['avg_f1_sell']:.4f} "
      f"(within-config fold-std {best['std_f1_sell']:.4f})")

print("\nLR baseline dict (paste into 2b model-comparison cell):")
print({
    "precision_buy":  round(all_results_df["avg_precision_buy"].mean(), 4),
    "recall_buy":     round(all_results_df["avg_recall_buy"].mean(), 4),
    "f1_buy":         round(all_results_df["avg_f1_buy"].mean(), 4),
    "precision_sell": round(all_results_df["avg_precision_sell"].mean(), 4),
    "recall_sell":    round(all_results_df["avg_recall_sell"].mean(), 4),
    "f1_sell":        round(all_results_df["avg_f1_sell"].mean(), 4),
})


PHASE 0 SUMMARY - LR, 24 configurations
Buy  F1: 0.2144 +/- 0.0811 across configs | median within-config fold-std 0.2230
Sell F1: 0.2455 +/- 0.1041 across configs | median within-config fold-std 0.2227

Best Sell-F1 config: ADA 3d fixed_0.5% -> Sell F1 0.4264 (within-config fold-std 0.2081)

LR baseline dict (paste into 2b model-comparison cell):
{'precision_buy': np.float64(0.2936), 'recall_buy': np.float64(0.2458), 'f1_buy': np.float64(0.2144), 'precision_sell': np.float64(0.3625), 'recall_sell': np.float64(0.2588), 'f1_sell': np.float64(0.2455)}


## SECTION 7: Feature Importance Analysis

Identify which on-chain metrics are most predictive of buy/sell/hold signals.

In [10]:
# ============================================================================
# FEATURE IMPORTANCE - LR Coefficients from Walk-Forward Validation
# ============================================================================
def compute_wfv_feature_importance(X, y, model_instance, param_dist, feature_cols, initial_train_size=0.6, step=30, n_iter=2, cv=3):
    """Compute LR feature importance using the same WFV + per-fold scaling protocol."""
    cont_idx, _ = _split_scale_columns(X)
    X_np = _to_np(X)
    y_np = np.asarray(y)
    fold_importances = []
    n = len(X_np)
    idx = int(initial_train_size * n)

    while idx < n:
        X_train_np, X_test_np = X_np[:idx], X_np[idx:idx + step]
        y_train_np = y_np[:idx]
        if len(X_test_np) == 0:
            break

        X_train_np, X_test_np = _scale_train_test(X_train_np, X_test_np, cont_idx)
        inner_cv = TimeSeriesSplit(n_splits=cv)
        search = RandomizedSearchCV(model_instance, param_dist, n_iter=n_iter, cv=inner_cv, scoring='f1_macro', random_state=42)
        try:
            search.fit(X_train_np, y_train_np)
            model = search.best_estimator_
            fold_importances.append(np.abs(model.coef_).mean(axis=0))
        except Exception:
            pass

        idx += step

    if not fold_importances:
        return None

    avg_importances = np.mean(fold_importances, axis=0)
    X_full = pd.DataFrame(X_np, columns=feature_cols)
    corr_with_signal = [X_full[col].corr(pd.Series(y_np).astype(float)) for col in feature_cols]

    return pd.DataFrame({
        'feature': feature_cols,
        'wfv_lr_importance': avg_importances,
        'signal_correlation': corr_with_signal
    }).sort_values('wfv_lr_importance', ascending=False)


for crypto, df, _feature_cols in [('BTC', btc_df, btc_feature_cols), ('ADA', ada_df, ada_feature_cols)]:
    print("\n" + "=" * 100)
    print(f"FEATURE IMPORTANCE (Walk-Forward Validation) - {crypto}")
    print("=" * 100)

    for h in horizons:
        for name, t in thresholds.items():
            signal_col = f"signal_{h}d_{name}"
            feature_cols_list = get_feature_columns(df)
            X = df[feature_cols_list]
            y = df[signal_col]
            combined = pd.concat([X, y], axis=1).dropna()
            X_clean = combined[feature_cols_list]
            y_clean = combined[signal_col].astype(int)

            if len(X_clean) < 100:
                continue

            lr_model = LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced')
            lr_param_dist = {'C': [0.1, 1, 10], 'solver': ['lbfgs']}

            feat_imp = compute_wfv_feature_importance(X_clean, y_clean, lr_model, lr_param_dist, feature_cols_list)

            if feat_imp is not None:
                print(f"\n{crypto} {h}d, {name}:")
                print(feat_imp[['feature', 'wfv_lr_importance', 'signal_correlation']].head(10).to_string(index=False))
            else:
                print(f"\n{crypto} {h}d, {name}: Could not compute (insufficient folds)")


FEATURE IMPORTANCE (Walk-Forward Validation) - BTC



BTC 3d, fixed_0.5%:
             feature  wfv_lr_importance  signal_correlation
          CapMVRVCur           0.472387           -0.003754
NVT_Tx_Basis_lag_30d           0.297235           -0.041242
      Puell_Multiple           0.279950            0.023138
 NVT_Tx_Basis_lag_7d           0.277904           -0.031094
     HashRate_30d_MA           0.267809            0.009703
           IssTotUSD           0.252055           -0.023141
 Price_Dist_lag_182d           0.201772           -0.007499
 NVT_Tx_Basis_lag_3d           0.181884           -0.032987
            HashRate           0.164887            0.017432
Tx_Intensity_lag_14d           0.152033           -0.004522



BTC 3d, fixed_1%:
             feature  wfv_lr_importance  signal_correlation
          CapMVRVCur           0.455356            0.002213
        NVT_Tx_Basis           0.215731           -0.036446
      Puell_Multiple           0.215501            0.030833
 Price_Dist_lag_182d           0.204197           -0.002521
            HashRate           0.188940            0.012833
 NVT_Tx_Basis_lag_7d           0.179175           -0.034761
           IssTotUSD           0.172973           -0.022158
NVT_Tx_Basis_lag_30d           0.155541           -0.045022
           AdrActCnt           0.145312            0.033764
    AdrActCnt_lag_7d           0.139999           -0.018771



BTC 3d, fixed_2%:
             feature  wfv_lr_importance  signal_correlation
          CapMVRVCur           0.475457            0.000821
      Puell_Multiple           0.244896            0.038646
 Price_Dist_lag_182d           0.218596            0.006182
NVT_Tx_Basis_lag_30d           0.208653           -0.071928
            HashRate           0.194416            0.002496
 Tx_Intensity_lag_7d           0.190181            0.019322
           IssTotUSD           0.171463           -0.032744
 NVT_Tx_Basis_lag_7d           0.161774           -0.060886
    AdrActCnt_lag_7d           0.144243           -0.015160
Tx_Intensity_lag_30d           0.137117           -0.013235



BTC 7d, fixed_0.5%:
             feature  wfv_lr_importance  signal_correlation
          CapMVRVCur           0.510392           -0.004786
NVT_Tx_Basis_lag_14d           0.447778           -0.071464
      Puell_Multiple           0.419348            0.025011
 Tx_Intensity_lag_7d           0.376131            0.028252
           IssTotUSD           0.270881           -0.052853
 NVT_Tx_Basis_lag_3d           0.239359           -0.065352
 NVT_Tx_Basis_lag_7d           0.237685           -0.063236
        NVT_Tx_Basis           0.231867           -0.066375
 Price_Dist_lag_182d           0.225627           -0.009009
Tx_Intensity_lag_30d           0.220420            0.012068


...


ADA 14d, fixed_1%:
             feature  wfv_lr_importance  signal_correlation
        NVT_Tx_Basis           0.572084            0.036971
 NVT_Tx_Basis_lag_1d           0.564190            0.037462
        StakingRatio           0.496097            0.022114
NVT_Tx_Basis_lag_14d           0.359029            0.021688
      ActiveStakeADA           0.341093           -0.017292
 NVT_Tx_Basis_lag_3d           0.335879            0.041422
          CapMVRVCur           0.321281           -0.009037
   AdrActCnt_lag_30d           0.292699           -0.133296
  Price_Dist_lag_90d           0.280477            0.104357
 Tx_Intensity_lag_3d           0.260590           -0.044172



ADA 14d, fixed_2%:
             feature  wfv_lr_importance  signal_correlation
 NVT_Tx_Basis_lag_1d           0.606700            0.036210
        StakingRatio           0.591392            0.027150
      ActiveStakeADA           0.441776           -0.013231
        NVT_Tx_Basis           0.388526            0.035353
 NVT_Tx_Basis_lag_3d           0.349841            0.039570
          CapMVRVCur           0.328258           -0.007633
NVT_Tx_Basis_lag_14d           0.277325            0.018476
NVT_Tx_Basis_lag_30d           0.274083           -0.000428
 NVT_Tx_Basis_lag_7d           0.269178            0.033978
  Price_Dist_lag_90d           0.246332            0.103794



ADA 30d, fixed_0.5%:
             feature  wfv_lr_importance  signal_correlation
        NVT_Tx_Basis           0.722396            0.087395
NVT_Tx_Basis_lag_30d           0.720756            0.022535
 NVT_Tx_Basis_lag_7d           0.663084            0.072798
    AdrActCnt_lag_7d           0.542660           -0.199080
  Price_Dist_lag_90d           0.491309            0.158026
        StakingRatio           0.489969           -0.030557
 NVT_Tx_Basis_lag_3d           0.476526            0.083764
    AdrActCnt_lag_1d           0.455180           -0.187480
          CapMVRVCur           0.436538            0.005996
NVT_Tx_Basis_lag_14d           0.415508            0.052698



ADA 30d, fixed_1%:
             feature  wfv_lr_importance  signal_correlation
NVT_Tx_Basis_lag_30d           0.720058            0.023617
        NVT_Tx_Basis           0.575688            0.090234
    AdrActCnt_lag_7d           0.543557           -0.203756
  Price_Dist_lag_90d           0.449219            0.162919
        StakingRatio           0.443352           -0.032551
Tx_Intensity_lag_30d           0.382460           -0.061492
 Tx_Intensity_lag_3d           0.365787           -0.065999
          CapMVRVCur           0.344918            0.007532
             SplyCur           0.304595           -0.125494
      ActiveStakeADA           0.277215           -0.087021



ADA 30d, fixed_2%:
             feature  wfv_lr_importance  signal_correlation
    AdrActCnt_lag_7d           0.514536           -0.209293
  Price_Dist_lag_90d           0.511664            0.159627
        StakingRatio           0.417188           -0.036241
          CapMVRVCur           0.369427            0.003784
NVT_Tx_Basis_lag_14d           0.359223            0.054873
Tx_Intensity_lag_30d           0.335203           -0.063350
 Tx_Intensity_lag_3d           0.311264           -0.067964
        NVT_Tx_Basis           0.275342            0.091933
 NVT_Tx_Basis_lag_1d           0.267483            0.091092
      ActiveStakeADA           0.265793           -0.090436


In [11]:
%%time
# =============================================================================
# FEATURE IMPORTANCE SUMMARY (averaged across all horizons/thresholds)
# =============================================================================
# Aggregates walk-forward LR importances to surface consistently strong features.
def summarize_feature_importance():
    summary = {}
    for crypto, df, _ in [('BTC', btc_df, btc_feature_cols), ('ADA', ada_df, ada_feature_cols)]:
        rows = []
        for h in horizons:
            for name, t in thresholds.items():
                signal_col = f'signal_{h}d_{name}'
                feature_cols_list = get_feature_columns(df)
                combined = pd.concat([df[feature_cols_list], df[signal_col]], axis=1).dropna()
                if len(combined) < 100:
                    continue
                X_clean = combined[feature_cols_list]
                y_clean = combined[signal_col].astype(int)
                lr_model = LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced')
                lr_param_dist = {'C': [0.1, 1, 10], 'solver': ['lbfgs']}
                feat_imp = compute_wfv_feature_importance(X_clean, y_clean, lr_model, lr_param_dist, feature_cols_list)
                if feat_imp is not None:
                    rows.append(feat_imp[['feature', 'wfv_lr_importance']].set_index('feature'))
        if rows:
            agg = pd.concat(rows, axis=1).mean(axis=1).sort_values(ascending=False)
            top10 = agg.head(10)
            summary[crypto] = top10
            print('\n' + '=' * 80)
            print(f'AVERAGED FEATURE IMPORTANCE - {crypto} (all horizons & thresholds)')
            print('=' * 80)
            print(top10.to_frame('avg_importance').to_string())
        else:
            print(f'No feature importance available for {crypto} (insufficient data)')
    return summary

fi_summary = summarize_feature_importance()


AVERAGED FEATURE IMPORTANCE - BTC (all horizons & thresholds)
                      avg_importance
feature                             
CapMVRVCur                  0.563206
Puell_Multiple              0.397592
IssTotUSD                   0.374726
Price_Dist_lag_182d         0.315974
HashRate_30d_MA             0.248930
NVT_Tx_Basis_lag_14d        0.247091
HashRate                    0.233638
NVT_Tx_Basis_lag_30d        0.211799
NVT_Tx_Basis_lag_7d         0.174904
NVT_Tx_Basis_lag_3d         0.168188



AVERAGED FEATURE IMPORTANCE - ADA (all horizons & thresholds)
                      avg_importance
feature                             
StakingRatio                0.340529
NVT_Tx_Basis                0.295500
NVT_Tx_Basis_lag_7d         0.289421
NVT_Tx_Basis_lag_1d         0.269358
NVT_Tx_Basis_lag_3d         0.265543
NVT_Tx_Basis_lag_30d        0.241826
NVT_Tx_Basis_lag_14d        0.239300
CapMVRVCur                  0.232804
Price_Dist_lag_90d          0.231570
ActiveStakeADA              0.220086
CPU times: total: 4min 11s
Wall time: 3min 58s


In [12]:
%%time
# ============================================================================
# BINARY MCC EVALUATION - USING WALK-FORWARD VALIDATION PREDICTIONS
# ============================================================================
print("Evaluating Binary MCC using Walk-Forward Validation predictions...")
print("Configuration: BTC, 30-day horizon, 1% fixed threshold\n")

sample_horizon = 30
sample_threshold = 0.01

btc_df[f"signal_{sample_horizon}d_sample"] = btc_df[f"fwd_return_{sample_horizon}d"].apply(
    lambda x: classify_signal(x, sample_threshold)
)

feature_cols_mcc = get_feature_columns(btc_df)

X_btc = btc_df[feature_cols_mcc]
y_btc = btc_df[f"signal_{sample_horizon}d_sample"]

combined = pd.concat([X_btc, y_btc], axis=1).dropna()
X_clean = combined[feature_cols_mcc]
y_clean = combined[f"signal_{sample_horizon}d_sample"].astype(int)

print(f"Dataset size: {len(X_clean)} samples")
print("Signal distribution (Training):")
print(f"  - Buy (1):  {(y_clean == 1).sum()} samples ({(y_clean == 1).sum()/len(y_clean)*100:.1f}%)")
print(f"  - Hold (0): {(y_clean == 0).sum()} samples ({(y_clean == 0).sum()/len(y_clean)*100:.1f}%)")
print(f"  - Sell (-1): {(y_clean == -1).sum()} samples ({(y_clean == -1).sum()/len(y_clean)*100:.1f}%)")

print("\nRunning walk-forward validation to collect predictions...")
lr_model = LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced')
lr_param_dist = {
    'C': [0.1, 1, 10],
    'solver': ['lbfgs']
}

avg_metrics_wfv, y_true_wfv, y_pred_wfv = walk_forward_with_predictions(
    X_clean, y_clean, lr_model, lr_param_dist,
    initial_train_size=0.6, step=30, n_iter=2, cv=3,
)

print("Walk-forward validation completed!")
print(f"Total predictions collected: {len(y_pred_wfv)} across all folds")

if len(y_pred_wfv) == 0:
    print("No predictions collected (insufficient test windows). Try reducing initial_train_size or the step size.")
else:
    print(f"\nWalk-Forward Prediction distribution:")
    print(f"  - Buy (1):  {sum(p == 1 for p in y_pred_wfv)} samples ({(sum(p == 1 for p in y_pred_wfv)/len(y_pred_wfv))*100:.1f}%)")
    print(f"  - Hold (0): {sum(p == 0 for p in y_pred_wfv)} samples ({(sum(p == 0 for p in y_pred_wfv)/len(y_pred_wfv))*100:.1f}%)")
    print(f"  - Sell (-1): {sum(p == -1 for p in y_pred_wfv)} samples ({(sum(p == -1 for p in y_pred_wfv)/len(y_pred_wfv))*100:.1f}%)")
    print("\n" + "=" * 80)
    print("BINARY MCC EVALUATION (Walk-Forward Validation)")
    print("=" * 80)
    btc_mcc_results = evaluate_signal_quality(y_true_wfv, y_pred_wfv, verbose=True)


Evaluating Binary MCC using Walk-Forward Validation predictions...
Configuration: BTC, 30-day horizon, 1% fixed threshold

Dataset size: 1949 samples
Signal distribution (Training):
  - Buy (1):  963 samples (49.4%)
  - Hold (0): 117 samples (6.0%)
  - Sell (-1): 869 samples (44.6%)

Running walk-forward validation to collect predictions...


Walk-forward validation completed!
Total predictions collected: 780 across all folds

Walk-Forward Prediction distribution:
  - Buy (1):  288 samples (36.9%)
  - Hold (0): 317 samples (40.6%)
  - Sell (-1): 175 samples (22.4%)

BINARY MCC EVALUATION (Walk-Forward Validation)

Binary MCC for Buy Signal (Class 1)
MCC: 0.0476
Weighted MCC components (Buy):
  [OK] TP: 150, [OK] TN: 260
  [X] Opposite FP (pred Buy | true Sell): 120 (x2.0)
  [X] Opposite FN (pred Sell | true Buy): 76 (x2.0)
  [X] Miss from Hold (true Buy | pred Hold): 156
  [X] False Buy from Hold (true Hold | pred Buy): 18
Weighted MCC (Buy): -0.1726

Binary MCC for Sell Signal (Class -1)
MCC: 0.0574
Weighted MCC components (Sell):
  [OK] TP: 86, [OK] TN: 349
  [X] Opposite FP (pred Sell | true Buy): 76 (x2.0)
  [X] Opposite FN (pred Buy | true Sell): 120 (x2.0)
  [X] Miss from Hold (true Sell | pred Hold): 136
  [X] False Sell from Hold (true Hold | pred Sell): 13
Weighted MCC (Sell): -0.1541
CPU times: total: 9.45 s
Wall 

In [13]:
# ============================================================================
# PHASE 0: PER-FOLD STABILITY (never trust a single pooled number)
# ============================================================================
# The pooled MCC above uses every walk-forward fold at once. Here we recompute
# the metrics fold-by-fold (chunking the collected predictions) and report
# mean +/- std. A std comparable to the mean means the signal is not stable.
from sklearn.metrics import f1_score as _f1, matthews_corrcoef as _mcc

if len(y_pred_wfv) > 0:
    _step = 30
    _yt = np.array(y_true_wfv)
    _yp = np.array(y_pred_wfv)
    _folds = [(_yt[i:i + _step], _yp[i:i + _step]) for i in range(0, len(_yt), _step)]
    _f1b, _f1s, _mb, _ms = [], [], [], []
    for _t, _p in _folds:
        _f1b.append(_f1(_t, _p, labels=[1], average='macro', zero_division=0))
        _f1s.append(_f1(_t, _p, labels=[-1], average='macro', zero_division=0))
        for _lab, _store in ((1, _mb), (-1, _ms)):
            _tb = (_t == _lab).astype(int)
            _pb = (_p == _lab).astype(int)
            if len(np.unique(_tb)) > 1 and len(np.unique(_pb)) > 1:
                _store.append(_mcc(_tb, _pb))
    print("\n" + "=" * 80)
    print("PER-FOLD STABILITY  (BTC 30d/1%, mean +/- std across walk-forward folds)")
    print("=" * 80)
    print(f"Folds: {len(_folds)} (step={_step})")
    print(f"Buy  F1 : {np.mean(_f1b):.4f} +/- {np.std(_f1b):.4f}")
    print(f"Sell F1 : {np.mean(_f1s):.4f} +/- {np.std(_f1s):.4f}")
    print(f"Buy  MCC: {np.mean(_mb):.4f} +/- {np.std(_mb):.4f}  (defined in {len(_mb)}/{len(_folds)} folds)")
    print(f"Sell MCC: {np.mean(_ms):.4f} +/- {np.std(_ms):.4f}  (defined in {len(_ms)}/{len(_folds)} folds)")
    print("NOTE: std comparable to the mean => signal not stable fold-to-fold;")
    print("      MCC undefined in folds where Buy or Sell is absent from the 30-day window.")



PER-FOLD STABILITY  (BTC 30d/1%, mean +/- std across walk-forward folds)
Folds: 26 (step=30)
Buy  F1 : 0.2964 +/- 0.2969
Sell F1 : 0.1856 +/- 0.2607
Buy  MCC: 0.0104 +/- 0.2300  (defined in 15/26 folds)
Sell MCC: 0.0923 +/- 0.1964  (defined in 12/26 folds)
NOTE: std comparable to the mean => signal not stable fold-to-fold;
      MCC undefined in folds where Buy or Sell is absent from the 30-day window.


## Logistic Regression: Final Analysis

> **Numbers refreshed 2026-06-22** from a clean end-to-end re-execution (leak-free per-fold scaling, no synthetic Hold labels, `TimeSeriesSplit` inner CV). Data: 2020-01-01 → 2026-06-01 (2344 rows/asset; 1949 usable for BTC 30d/1% after `dropna`). Seeds pinned (`random_state=42`).

### Binary MCC — BTC 30d / 1% threshold

| Metric | Value | Interpretation |
|--------|-------|----------------|
| Buy MCC | 0.0476 | Near-random (0 = coin flip) |
| Buy Weighted MCC | −0.1726 | Negative — opposite-direction errors dominate |
| Sell MCC | 0.0574 | Near-random |
| Sell Weighted MCC | −0.1541 | Negative — opposite-direction errors dominate |

### Never one number — per-fold stability (BTC 30d/1%, 26 walk-forward folds)

| Metric | Mean ± std | Folds defined |
|--------|-----------|---------------|
| Buy F1  | 0.296 ± 0.297 | 26/26 |
| Sell F1 | 0.186 ± 0.261 | 26/26 |
| Buy MCC  | 0.010 ± 0.230 | 15/26 |
| Sell MCC | 0.092 ± 0.196 | 12/26 |

**The std is as large as the mean**, and MCC is undefined in ~40–50% of folds (Buy or Sell absent from the 30-day window). Fold-to-fold, the signal is indistinguishable from noise — a single pooled number overstates how much is really there.

### F1 across 24 configurations

- Buy F1: **0.214 ± 0.081** (across configs) · Sell F1: **0.246 ± 0.104**
- Best Sell-F1 config: **ADA 3d / 0.5% → 0.426** (within-config fold-std 0.208) — treat as a best-of-24 candidate, not a confirmed edge (multiple-testing; see `VALIDATION_PLAN.md` Phase 2.3).

### Why LR is a weak baseline

Linear decision boundaries can't capture the non-linear, regime-dependent interactions between NVT, Tx_Intensity, and momentum features. LR is a deliberate **baseline** — it sets the floor that XGBoost (`2b`) must clear. At 1%/30d, Hold is only ~6% of labels, so the "3-class" task is effectively Buy-vs-Sell.

### Next step
This is Phase 0 (trustworthy, reproducible numbers). The follow-up validation — price-only vs +on-chain ablation and model-free signal detection — has since been completed; the verdict (no on-chain directional edge over price) is in `docs/CONCLUSIONS.md`.